# Project 1 · Step 2 — Yield model (Nordsøen Midt & Hesselø)

**Goal:** an hourly potential-output series per site that acts as

* the proxy for **Available Active Power (AAP′)** – the settlement volume of the capability-based CfD (contract_assumptions.md rows B1–B3, B12), and
* the **metered output at the point of connection** – the volume sold on the day-ahead market (B13).

**Inputs**
* `data/era5_NSM_<y0>_<y1>.csv`, `data/era5_HES_<y0>_<y1>.csv` – from `era5_download.py`
* `data/IEA_Reference_15MW_240.csv` – IEA 15 MW reference turbine power & thrust curve (NREL `turbine-models` repository)

**Outputs** (in `outputs/`)
* `yield_hourly_NSM.csv.gz`, `yield_hourly_HES.csv.gz` – hourly series, UTC
* `yield_annual_summary.csv` – per site and climate year
* `yield_config.json` – every assumption used, for the memo and for step 3

**Scope:** first-pass P50 only. Wake losses are a stated percentage here; PyWake modelling is Project 2. Curtailment in negative-price hours is *not* applied here – that is a dispatch decision in step 3 and does not change AAP (row C1).

Every assumption carries a status: **Doc** (tender documents), **Ext** (external primary/secondary source), **Asm** (my modelling assumption).

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = Path("data")
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

## 1 · Assumptions

| Item | NSM | HES | Status / source |
|---|---|---|---|
| Representative point | 56.00 N, 7.55 E | 56.352 N, 11.698 E | NSM: **Asm** – point in Energinet sub-area A2 (Nordsøen I Midt), ~20 km off Hvide Sande; replace with Appendix 2 centroid. HES: **Ext** – centroid of HS-A/B/C, EMD *Hesselø South Site Wind Conditions Assessment* rev. 3 (Dec 2025), Table 25 |
| Turbine | IEA 15 MW RWT, D = 240 m, hub 150 m | same | **Asm** – public reference turbine; 150 m is also the reference height of Energinet's wind studies |
| Turbines / MinCap | 67 × 15 MW = 1,005 MW | 53 × 15 MW = 795 MW | **Doc** rounding rule (Annex F 2.32): MinCap baseline ÷ turbine size, rounded to a whole turbine at the developer's discretion; here rounded to nearest. Hesselø baseline 800 MW |
| Shear 100 → 150 m | α = 0.07 | α = 0.09 | **Ext** – NSI: C2Wind *Lot 3 (North Sea I) Wind Assessment Suppl. Note* rev. 3 (Nov 2024) §6.2.1; HES: EMD report Table 1 |
| Long-term mean at 150 m (calibration target) | 10.44 m/s | 9.63 m/s | **Ext** – NSI: C2Wind Table 6-1, analysis point A2 P2 (Weibull mean; A = 11.76, k = 2.25). HES: mean of HS-1-LB, HS-A, HS-B, HS-C arithmetic means (9.60, 9.67, 9.58, 9.65), EMD Tables 17 & 26. Both are 12-month floating-LiDAR campaigns long-term corrected over ~22–23 years |
| Internal wake loss | 10 % | 10 % | **Ext/Asm** – DTU report E-0255 (2025, for the DEA): domain-average wake losses 6.6–12 % for today's Danish fleet and 12–21 % for the 2030 build-out scenario (incl. cluster effects). 10 % taken for the internal part of a single ~1 GW farm |
| External wake | 1.0 % (Thor, ~40 km NNE) | 0.3 % (Anholt, ~26 km NW) | **Asm** – NSM: Thor lies in a low-frequency direction; HES: EMD's WRF run found Anholt cuts mean wind speed by <0.1 % over most of the area and ≤0.4 % at the NW edge. Nordsøen Syd (from ~2035) and a future Kattegat farm are scenarios, not base case |
| Other performance losses | 2 % | 2 % | **Asm** – power-curve under-performance, blockage, hysteresis |
| Unscheduled unavailability | 3 % | 3 % | **Asm** – contract_assumptions B3. Excluded from AAP′ |
| Scheduled unavailability | 1 % | 1 % | **Asm** – counted in AAP′ (pre-notified on REMIT IIP) but not produced |
| Electrical losses to POC | 2.5 % | 2.5 % | **Asm** – array + export + transformer losses. AAP is measured at turbine level and is **not** corrected for grid losses (App 3.1 §3.2), so this only hits metered output |

In [ ]:
CFG = {
    "turbine": {"name": "IEA 15 MW RWT", "rated_MW": 15.0, "rotor_m": 240, "hub_m": 150,
                "cut_out_ms": 25.0, "curve_file": "IEA_Reference_15MW_240.csv"},
    "sites": {
        "NSM": {"name": "Nordsøen Midt", "zone": "DK1", "lat": 56.00, "lon": 7.55,
                "n_turbines": 67, "alpha": 0.07, "target_mean_150": 10.44,
                "wake_internal": 0.10, "wake_external": 0.010},
        "HES": {"name": "Hesselø", "zone": "DK2", "lat": 56.352, "lon": 11.698,
                "n_turbines": 53, "alpha": 0.09, "target_mean_150": 9.63,
                "wake_internal": 0.10, "wake_external": 0.003},
    },
    "losses": {"other_performance": 0.02, "unscheduled_unavail": 0.03,
               "scheduled_unavail": 0.01, "electrical": 0.025},
    "options": {
        "shear_method": "site_alpha",   # "site_alpha" (constant, Energinet) or "era5_alpha" (hourly 10→100 m)
        "calibrate": True,              # scale ERA5 so the long-term mean matches Energinet's LiDAR-based value
        "density_correction": False,    # IEC-style wind-speed adjustment (rho/1.225)^(1/3)
        "wake_shape": "ct_weighted",    # "flat" or "ct_weighted" (bigger losses where thrust is high)
    },
    "dea_check": {"NSM_GWh_per_yr_per_GW": 4479,
                  "NSM_monthly_GWh": [473, 424, 437, 290, 321, 399, 323, 389, 495, 330, 233, 366],
                  "NSS_monthly_GWh": [537, 430, 454, 291, 270, 277, 267, 323, 342, 461, 463, 369],
                  "source": "DEA Step-by-Step Guide Table 1.5 (NSM); Appendix 3 Table 1 (NSS)"},
}
for s in CFG["sites"].values():
    s["mincap_MW"] = s["n_turbines"] * CFG["turbine"]["rated_MW"]
pd.DataFrame(CFG["sites"]).T

## 2 · Power curve

In [ ]:
pc = pd.read_csv(DATA / CFG["turbine"]["curve_file"]).iloc[:, :5]
pc.columns = ["ws", "power_kW", "cp", "thrust_kN", "ct"]
pc = pc.dropna().sort_values("ws")
CUT_OUT = CFG["turbine"]["cut_out_ms"]

def turbine_power_MW(ws):
    p = np.interp(ws, pc.ws, pc.power_kW, left=0.0, right=0.0) / 1000.0
    return np.where((ws < pc.ws.min()) | (ws > CUT_OUT), 0.0, p)

def turbine_ct(ws):
    c = np.interp(ws, pc.ws, pc.ct, left=0.0, right=0.0)
    return np.where(ws > CUT_OUT, 0.0, c)

v = np.linspace(0, 30, 301)
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.plot(v, turbine_power_MW(v), label="Power [MW]")
ax2 = ax.twinx(); ax2.plot(v, turbine_ct(v), color="tab:orange", alpha=.7, label="Ct")
ax.set_xlabel("Hub-height wind speed [m/s]"); ax.set_ylabel("MW"); ax2.set_ylabel("Ct")
ax.set_title("IEA 15 MW reference turbine (hard cut-out at 25 m/s)"); fig.tight_layout()
print(f"Rated power in table: {pc.power_kW.max()/1000:.3f} MW; cut-in {pc.ws.min():.1f} m/s")

## 3 · Load ERA5 and check coverage

In [ ]:
def load_era5(site):
    files = sorted(DATA.glob(f"era5_{site}_*.csv"))
    if not files:
        raise FileNotFoundError(f"No data/era5_{site}_*.csv – run era5_download.py first")
    f = max(files, key=lambda p: p.stat().st_size)
    df = pd.read_csv(f, parse_dates=["time_utc"]).set_index("time_utc").sort_index()
    df = df[~df.index.duplicated()]
    df["ws100"] = np.hypot(df.u100, df.v100)
    df["ws10"] = np.hypot(df.u10, df.v10)
    df["wd100"] = (270 - np.degrees(np.arctan2(df.v100, df.u100))) % 360  # meteorological: direction wind comes FROM
    print(f"{site}: {f.name}  {df.index.min():%Y-%m-%d} → {df.index.max():%Y-%m-%d}  rows={len(df):,}")
    return df

era5 = {s: load_era5(s) for s in CFG["sites"]}

cov = []
for s, df in era5.items():
    full = pd.date_range(df.index.min().floor("D"), df.index.max().ceil("D") - pd.Timedelta("1h"), freq="h")
    cov.append({"site": s, "years": df.index.year.nunique(), "missing_h": len(full.difference(df.index)),
                "mean_ws10": df.ws10.mean(), "mean_ws100": df.ws100.mean()})
cov = pd.DataFrame(cov).set_index("site"); cov

## 4 · Extrapolate to hub height (150 m) and calibrate

ERA5 is a ~31 km reanalysis and is known to under-state offshore wind speeds a little. Energinet's site studies long-term-correct 12 months of floating LiDAR at 150 m, so their mean is the better anchor. The calibration is a single multiplicative factor on the hourly hub-height speed (keeps ERA5's timing and shape, fixes the level). Both raw and calibrated results are reported.

In [ ]:
RHO0 = 1.225

def hub_speed(df, site_cfg):
    hub = CFG["turbine"]["hub_m"]
    if CFG["options"]["shear_method"] == "era5_alpha":
        a = np.log(df.ws100.clip(lower=0.5) / df.ws10.clip(lower=0.5)) / np.log(100 / 10)
        a = a.clip(-0.1, 0.4)
    else:
        a = site_cfg["alpha"]
    ws = df.ws100 * (hub / 100) ** a
    if CFG["options"]["density_correction"] and {"t2m_K", "sp_Pa"} <= set(df.columns):
        T_hub = df.t2m_K - 0.0065 * (hub - 2)
        p_hub = df.sp_Pa * np.exp(-9.81 * hub / (287.05 * T_hub))
        rho = p_hub / (287.05 * T_hub)
        ws = ws * (rho / RHO0) ** (1 / 3)
    return ws

calib = []
for s, df in era5.items():
    sc = CFG["sites"][s]
    df["ws_hub_raw"] = hub_speed(df, sc)
    factor = sc["target_mean_150"] / df.ws_hub_raw.mean() if CFG["options"]["calibrate"] else 1.0
    df["ws_hub"] = df.ws_hub_raw * factor
    sc["calibration_factor"] = float(factor)
    calib.append({"site": s, "ERA5 mean @150m (raw)": df.ws_hub_raw.mean(),
                  "Energinet LT mean @150m": sc["target_mean_150"], "scale factor": factor})
pd.DataFrame(calib).set_index("site")

## 5 · Gross output and loss stack

In [ ]:
def wake_loss_series(ws, gross_MW, L):
    # Hourly wake loss fraction with the annual energy-weighted mean equal to L.
    if CFG["options"]["wake_shape"] == "flat" or L == 0:
        return pd.Series(L, index=ws.index)
    w = pd.Series(turbine_ct(ws.values), index=ws.index)
    k = L * gross_MW.sum() / (gross_MW * w).sum()
    return (k * w).clip(upper=0.6)

res = {}
for s, df in era5.items():
    sc, Ls = CFG["sites"][s], CFG["losses"]
    out = pd.DataFrame(index=df.index)
    out["ws_hub"] = df.ws_hub
    out["wd100"] = df.wd100
    out["gross_MW"] = sc["n_turbines"] * turbine_power_MW(df.ws_hub.values)
    out["gross_raw_MW"] = sc["n_turbines"] * turbine_power_MW(df.ws_hub_raw.values)
    out["wake_int"] = wake_loss_series(df.ws_hub, out.gross_MW, sc["wake_internal"])
    out["wake_ext"] = wake_loss_series(df.ws_hub, out.gross_MW, sc["wake_external"])
    potential = out.gross_MW * (1 - out.wake_int) * (1 - out.wake_ext) * (1 - Ls["other_performance"])
    out["potential_MW"] = potential.clip(upper=sc["mincap_MW"])          # turbine-level potential, all turbines available
    out["aap_proxy_MWh"] = out.potential_MW * (1 - Ls["unscheduled_unavail"])   # CfD volume (AAP′), 1 h steps
    out["metered_MWh"] = (out.potential_MW
                          * (1 - Ls["unscheduled_unavail"] - Ls["scheduled_unavail"])
                          * (1 - Ls["electrical"]))                       # sold at POC
    res[s] = out

def waterfall(s):
    o, sc, Ls = res[s], CFG["sites"][s], CFG["losses"]
    yrs = o.index.year.nunique() if len(o) > 8000 else len(o) / 8760
    per_yr = lambda x: x.sum() / 1000 / yrs  # GWh/yr
    g = per_yr(o.gross_MW)
    steps = [("Gross (calibrated ERA5 × power curve)", g),
             ("after internal wake", per_yr(o.gross_MW * (1 - o.wake_int))),
             ("after external wake", per_yr(o.gross_MW * (1 - o.wake_int) * (1 - o.wake_ext))),
             ("after other performance", per_yr(o.potential_MW)),
             ("AAP′ proxy (after unscheduled unavail.)", per_yr(o.aap_proxy_MWh)),
             ("Metered at POC (after scheduled unavail. + electrical)", per_yr(o.metered_MWh))]
    t = pd.DataFrame(steps, columns=["step", "GWh/yr"]).set_index("step")
    t["CF on MinCap"] = t["GWh/yr"] * 1000 / (sc["mincap_MW"] * 8760)
    t["% of gross"] = t["GWh/yr"] / g
    return t

for s in res:
    print(f"\n=== {CFG['sites'][s]['name']}  (MinCap {CFG['sites'][s]['mincap_MW']:.0f} MW) ===")
    display(waterfall(s))

## 6 · Inter-annual variability and seasonality

In [ ]:
rows = []
for s, o in res.items():
    mc = CFG["sites"][s]["mincap_MW"]
    a = o.groupby(o.index.year)[["gross_MW", "aap_proxy_MWh", "metered_MWh"]].sum() / 1000
    a.columns = ["gross_GWh", "aap_proxy_GWh", "metered_GWh"]
    a["aap_CF"] = a.aap_proxy_GWh * 1000 / (mc * 8760)
    a["mean_ws_hub"] = o.groupby(o.index.year).ws_hub.mean()
    a["hours"] = o.groupby(o.index.year).size()
    a = a[a.hours >= 8700]                       # complete years only
    a.insert(0, "site", s); rows.append(a)
annual = pd.concat(rows)
annual.index.name = "year"

summ = annual.groupby("site").agg(years=("aap_proxy_GWh", "size"),
                                  aap_P50_GWh=("aap_proxy_GWh", "mean"),
                                  aap_std_GWh=("aap_proxy_GWh", "std"),
                                  metered_GWh=("metered_GWh", "mean"),
                                  aap_CF=("aap_CF", "mean"))
summ["IAV_%"] = 100 * summ.aap_std_GWh / summ.aap_P50_GWh
display(summ)

fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
for s, o in res.items():
    annual.loc[annual.site == s, "aap_CF"].plot(ax=axes[0], marker="o", label=s)
    m = o.aap_proxy_MWh.groupby(o.index.month).sum() / o.index.year.nunique() / 1000
    (m / CFG["sites"][s]["mincap_MW"] * 1000).plot(ax=axes[1], marker="o", label=f"{s} model (per GW)")
axes[1].plot(range(1, 13), CFG["dea_check"]["NSM_monthly_GWh"], "k--", label="DEA NSM interim estimate")
axes[1].plot(range(1, 13), CFG["dea_check"]["NSS_monthly_GWh"], "k:", label="DEA NSS interim estimate")
axes[0].set_title("Annual AAP′ capacity factor by climate year"); axes[0].legend()
axes[1].set_title("Monthly AAP′, GWh per GW (UTC months)"); axes[1].set_xlabel("month"); axes[1].legend(fontsize=8)
fig.tight_layout()

## 7 · Check against the DEA's Nordsøen Midt estimate

The DEA uses **4,479 GWh/yr for 1 GW** (≈51.1 % capacity factor) to settle the CfD before the AAP table is approved (Guide Table 1.5; contract_assumptions B11). The DEA does not say whether this is turbine-level potential or metered output, so both are compared. A difference within ±5 % is treated as consistent.

In [ ]:
DEA = CFG["dea_check"]["NSM_GWh_per_yr_per_GW"]
o, sc = res["NSM"], CFG["sites"]["NSM"]
yrs = annual[annual.site == "NSM"]
per_GW = lambda col: yrs[col].mean() * 1000 / sc["mincap_MW"]
gross_per_GW_raw = (o.gross_raw_MW.groupby(o.index.year).sum() / 1000).loc[yrs.index].mean() * 1000 / sc["mincap_MW"]

chk = pd.DataFrame({
    "GWh/yr per GW": {
        "Gross, raw ERA5": gross_per_GW_raw,
        "Gross, calibrated": per_GW("gross_GWh"),
        "AAP′ proxy": per_GW("aap_proxy_GWh"),
        "Metered at POC": per_GW("metered_GWh"),
        "DEA estimate": DEA}})
chk["CF"] = chk["GWh/yr per GW"] / 8760
chk["vs DEA"] = chk["GWh/yr per GW"] / DEA - 1
display(chk)

implied_loss = 1 - DEA / per_GW("gross_GWh")
print(f"DEA figure implies total losses of {implied_loss:.1%} from our calibrated gross "
      f"(our stack: {1 - per_GW('metered_GWh')/per_GW('gross_GWh'):.1%} to metered, "
      f"{1 - per_GW('aap_proxy_GWh')/per_GW('gross_GWh'):.1%} to AAP′).")
for name in ["AAP′ proxy", "Metered at POC"]:
    d = chk.loc[name, "vs DEA"]
    flag = "OK (within ±5 %)" if abs(d) <= 0.05 else ("HIGH – see guidance" if d > 0 else "LOW – see guidance")
    print(f"{name:16s} {d:+.1%}  {flag}")

### If the result is far from 4,479 GWh/yr — what to adjust, in order

1. **Check the wind level first.** Compare *ERA5 mean @150 m (raw)* with Energinet's 10.44 m/s. ERA5 is typically a few per cent low offshore; a raw gap above ~8 % suggests the wrong grid point, a shear problem (try `shear_method="era5_alpha"`), or a download/units error. Yield moves roughly 1.5–2 × the percentage change in mean wind speed at these sites.
2. **Our result is high (> +5 %).** Most likely the wake assumption is light: the DTU 2030 scenario gives 12–21 % total wake loss, so test 13–15 % internal plus 2–3 % external (Thor + future Nordsøen Syd). Also consider that the DEA may have assumed a lower specific-power or lower hub-height turbine, or metered output with higher availability losses (5–6 %).
3. **Our result is low (< −5 %).** Check that the calibration is switched on, that the turbine count gives ~1 GW, and that complete years are being used. Then revisit losses (8 % internal wake is defensible for a well-spaced layout) and consider a larger modern turbine (higher hub, larger rotor).
4. **Don't force a perfect match.** The DEA figure is an interim settlement estimate, not a P50 energy assessment, and its monthly profile looks like a single year. Record the gap and the reason in the memo; in step 5 the bid calibration will show how much a ±5 % yield change moves the implied capex/WACC.
5. **Hesselø has no DEA estimate** in the documents we hold. Use the NSM check to validate the method, then rely on EMD's wind level for Hesselø.

## 8 · Save outputs

In [ ]:
for s, o in res.items():
    cols = ["ws_hub", "wd100", "gross_MW", "wake_int", "wake_ext", "potential_MW", "aap_proxy_MWh", "metered_MWh"]
    o[cols].round(4).to_csv(OUT / f"yield_hourly_{s}.csv.gz", index_label="time_utc", compression="gzip")
annual.round(4).to_csv(OUT / "yield_annual_summary.csv")
with open(OUT / "yield_config.json", "w") as f:
    json.dump(CFG, f, indent=2, default=float)
print("Saved:", *sorted(p.name for p in OUT.iterdir()), sep="\n  ")

**Notes for step 3 (prices):** timestamps are UTC hours (ERA5 instantaneous values on the hour). Day-ahead prices are published in CET/CEST and, since Oct 2025, in 15-minute MTUs – convert prices to hourly UTC before joining. The hourly series cover the ERA5 climate years, not calendar operating years; the cash-flow model should map them onto 2033–2052 (e.g. average year, or cycling the climate years).